In [1]:
import os
import sys

import pickle

import torch

import pandas as pd

import pm4py
from pm4py.visualization.petri_net import visualizer as pn_vis

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from process.engine import ProcessModelConstraintEngine
from process.constraint import ProcessConstraint

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper
from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
    path="../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'A_FINALIZED', 'O_SELECTED'},
 {'A_DECLINED', 'O_DECLINED'},
 {'A_ACTIVATED', 'A_APPROVED', 'A_REGISTERED', 'O_ACCEPTED'}]

In [16]:
engine.branching_sets

[{'A_CANCELLED',
  'A_FINALIZED',
  'O_CREATED',
  'O_SELECTED',
  'O_SENT',
  'O_SENT_BACK'},
 {'A_ACTIVATED',
  'A_APPROVED',
  'A_DECLINED',
  'A_REGISTERED',
  'O_ACCEPTED',
  'O_DECLINED'}]

### --- Counterfactuals ---

In [17]:
dice4el_config_equal = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
dice4el_config_equal.validate()

dice4el_config_weighted = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=3.0,
    w_process_violation=3.0
)
dice4el_config_weighted.validate()

In [18]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-cf_domain_examples_dice4el_output.txt", console=False)

In [19]:
with open("../../experiments/cf_domain_examples.pkl", "rb") as f:
    data = pickle.load(f)

experiments = data["experiments"]
metadata = data["metadata"]

In [20]:
def dice4el_search_for_experiment(
    engine,
    experiment,
    exp_name,
    dice4el_config,
    train_arr,
    change_allowed_positions,
    activity_idx,
    user_rules
):

    results = []

    candidate_groups = experiment["candidates"]
    conf_arr = experiment["conformant"]

    for group_idx, cand_arr in enumerate(candidate_groups):

        print(
            f"Running experiment: {exp_name} "
            f"(group {group_idx + 1}) | cases: {cand_arr.shape[0]}"
        )

        for i in range(cand_arr.shape[0]):

            trace = cand_arr[i]
            trace_activities = [row[activity_idx] for row in trace]
            
            flexible_constraints = engine.generate_flexible_constraints(trace_activities)
            print("Flexible Constraints:", flexible_constraints)
                    
            desired_constraints = engine.generate_desired_constraints(
                trace_activities=trace_activities,
                user_specs=user_rules
            )
            print("Desired Constraints: ", desired_constraints)

            dice4el = EventLogDiCEOptimized(
                dice4el_config=dice4el_config,
                next_event_model_wrapper=model_wrapper,
                scenario_model_wrapper=scenario_model_wrapper,
            )

            cf = dice4el.search(
                trace=trace,
                train_arr=train_arr,
                conformant_arr=conf_arr,
                change_allowed_positions=change_allowed_positions,
                desired_constraints=desired_constraints,
                flexible_constraints=flexible_constraints,
                output_path=(
                    f"./counterfactuals/{exp_name}_group_{group_idx + 1}_case_{i}.xlsx"
                )
            )

            results.append(cf["eval"])

    return results

#### 1. A_ACCEPTED Milestone

In [21]:
experiment_a_preaccepted = experiments["a_preaccepted"]

change_allowed_positions = [0, 1]
user_rules_a_preaccepted = {
    ("A_DECLINED", 1): {"A_PREACCEPTED"},
    ("A_CANCELLED", 1): {"A_PREACCEPTED"},
}

train_arr_a_preaccepted = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_preaccepted"],
    sort_field="time:timestamp"
)

results_a_preaccepted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_a_preaccepted,
    exp_name="a_preaccepted",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_preaccepted,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_preaccepted
)

In [22]:
experiment_a_accepted = experiments["a_accepted"]

change_allowed_positions = [0, 1, 2]
user_rules_a_accepted = {
    ("A_DECLINED", 1): {"A_ACCEPTED"},
    ("A_CANCELLED", 1): {"A_ACCEPTED"},
}

train_arr_a_accepted = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_accepted"],
    sort_field="time:timestamp"
)

results_a_accepted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_a_accepted,
    exp_name="a_accepted",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_accepted,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_accepted
)

#### 2. A_FINALIZED Milestone

In [23]:
experiment_a_finalized = experiments["a_finalized"]

change_allowed_positions = [0, 1, 2, 3]
user_rules_a_finalized = {
    ("A_DECLINED", 1): {"A_FINALIZED", "O_SELECTED"},
    ("A_CANCELLED", 1): {"A_FINALIZED", "O_SELECTED"},
}

train_arr_a_finalized = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_finalized"],
    sort_field="time:timestamp"
)

results_a_finalized = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_a_finalized,
    exp_name="a_finalized",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_finalized,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_finalized
)

#### 3. A_APPROVED Milestone

In [24]:
experiment_1_a_approved = experiments["1_a_approved"]

change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7]
user_rules_1_a_approved = {
    ("A_CANCELLED", 1): {"O_SENT_BACK"},
    ("O_CANCELLED", 1): {"A_APPROVED", "O_ACCEPTED"},
}

train_arr_1_a_approved = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["1_a_approved"],
    sort_field="time:timestamp"
)

results_1_a_approved_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_1_a_approved,
    exp_name="1_a_approved_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_a_approved
)
results_1_a_approved_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_1_a_approved,
    exp_name="1_a_approved_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_a_approved
)

experiment_2_a_approved = experiments["2_a_approved"]

change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7, 8]
user_rules_2_a_approved = {
    ("A_DECLINED", 1): {"A_APPROVED"},
    ("O_DECLINED", 1): {"O_ACCEPTED"},
}

train_arr_2_a_approved = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["2_a_approved"],
    sort_field="time:timestamp"
)

results_2_a_approved_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_2_a_approved,
    exp_name="2_a_approved_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_a_approved
)
results_2_a_approved_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_2_a_approved,
    exp_name="2_a_approved_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_a_approved
)

#### 4. Swapped Ordering of A_APPROVED A_REGISTERED

In [25]:
change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

experiment_1_swap = experiments["1_swap"]

user_rules_1_swap = {
    ("A_REGISTERED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_1_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["1_swap"],
    sort_field="time:timestamp"
)

results_1_swap_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_1_swap,
    exp_name="1_swap_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_swap
)
results_1_swap_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiments["1_swap"],
    exp_name="1_swap_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_swap
)

experiment_2_swap = experiments["2_swap"]

user_rules_2_swap = {
    ("A_ACTIVATED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_ACTIVATED"},
}

train_arr_2_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["2_swap"],
    sort_field="time:timestamp"
)

results_2_swap_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_2_swap,
    exp_name="2_swap_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_swap
)
results_2_swap_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_2_swap,
    exp_name="2_swap_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_swap
)

experiment_3_swap = experiments["3_swap"]

user_rules_3_swap = {
    ("A_REGISTERED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_3_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["3_swap"],
    sort_field="time:timestamp"
)

results_3_swap_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_3_swap,
    exp_name="3_swap_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_3_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_3_swap
)
results_3_swap_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_3_swap,
    exp_name="3_swap_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_3_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_3_swap
)

experiment_4_swap = experiments["4_swap"]

user_rules_4_swap = {
    ("A_ACTIVATED", 1): {"A_APPROVED"},
    ("A_REGISTERED", 1): {"A_ACTIVATED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_4_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["4_swap"],
    sort_field="time:timestamp"
)

results_4_swap_equal = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_4_swap,
    exp_name="4_swap_equal",
    dice4el_config=dice4el_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_4_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_4_swap
)
results_4_swap_weighted = dice4el_search_for_experiment(
    engine=engine,
    experiment=experiment_4_swap,
    exp_name="4_swap_weighted",
    dice4el_config=dice4el_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_4_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_4_swap
)

### --- Overall Evaluation ---

In [26]:
def calculate_summary_eval(overall_results):
    df = pd.DataFrame(overall_results)

    summary_evaluation_df = pd.DataFrame({
        "Metric": df.columns,
        "Mean": df.mean(numeric_only=True).values,
        "Std": df.std(numeric_only=True).values,
    })

    return summary_evaluation_df


def analyze_overall_results(overall_results, group_name: str):

    os.makedirs("results", exist_ok=True)

    # Flatten experiments
    all_results = [
        result
        for experiment in overall_results
        for result in experiment
    ]

    summary_eval_df = calculate_summary_eval(all_results)

    with pd.ExcelWriter(
        f"results/summary_evaluation_{group_name}.xlsx",
        engine="openpyxl"
    ) as writer:

        summary_eval_df.to_excel(
            writer,
            sheet_name="summary_evaluation",
            index=False
        )

    return summary_eval_df

In [27]:
analyze_overall_results([results_a_preaccepted], group_name="a_preaccepted")

,Metric,Mean,Std
0,DISTANCE,0.160176,0.123721
1,CONT_DISTANCE,0.083989,0.048084
2,CAT_DISTANCE,0.236364,0.233550
3,SPARSITY,0.397727,0.145969
4,PROCESS_VIOLATION,0.000000,0.000000
5,MARGIN_LOSS,0.265115,0.140685
6,MARGIN_LOSS_NA,0.265115,0.140685
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.000000,0.000000
9,FITNESS,0.823019,0.311383


In [28]:
analyze_overall_results([results_a_accepted], group_name="a_accepted")

,Metric,Mean,Std
0,DISTANCE,0.142444,0.130199
1,CONT_DISTANCE,0.184887,0.109469
2,CAT_DISTANCE,0.100000,0.190414
3,SPARSITY,0.360000,0.114248
4,PROCESS_VIOLATION,0.000000,0.000000
5,MARGIN_LOSS,0.068009,0.140720
6,MARGIN_LOSS_NA,0.121625,0.176104
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.000000,0.000000
9,FITNESS,0.570453,0.259723


In [29]:
analyze_overall_results([results_a_finalized], group_name="a_finalized")

,Metric,Mean,Std
0,DISTANCE,0.131035,0.053859
1,CONT_DISTANCE,0.254927,0.095395
2,CAT_DISTANCE,0.007143,0.031944
3,SPARSITY,0.333333,0.027037
4,PROCESS_VIOLATION,0.007692,0.034401
5,MARGIN_LOSS,0.000685,0.003062
6,MARGIN_LOSS_NA,0.000685,0.003062
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.000000,0.000000
9,FITNESS,0.472745,0.077537


In [30]:
analyze_overall_results([results_1_a_approved_equal, results_2_a_approved_equal], group_name="a_approved_equal")

,Metric,Mean,Std
0,DISTANCE,0.323974,0.194704
1,CONT_DISTANCE,0.320771,0.101153
2,CAT_DISTANCE,0.327178,0.321905
3,SPARSITY,0.548807,0.177562
4,PROCESS_VIOLATION,0.147493,0.196043
5,MARGIN_LOSS,0.347005,0.336671
6,MARGIN_LOSS_NA,0.314601,0.345113
7,MARGIN_LOSS_FLIPPED,0.300000,0.425455
8,MARGIN_LOSS_NA_FLIPPED,0.243750,0.420922
9,FITNESS,1.367279,0.824353


In [31]:
analyze_overall_results([results_1_a_approved_weighted, results_2_a_approved_weighted], group_name="a_approved_weighted")

,Metric,Mean,Std
0,DISTANCE,0.318188,0.192572
1,CONT_DISTANCE,0.327948,0.104232
2,CAT_DISTANCE,0.308428,0.315502
3,SPARSITY,0.543125,0.172083
4,PROCESS_VIOLATION,0.148764,0.213274
5,MARGIN_LOSS,0.343173,0.322671
6,MARGIN_LOSS_NA,0.318092,0.337783
7,MARGIN_LOSS_FLIPPED,0.275000,0.404875
8,MARGIN_LOSS_NA_FLIPPED,0.237500,0.413192
9,FITNESS,2.337124,1.813450


In [32]:
analyze_overall_results([results_1_swap_equal, results_2_swap_equal, results_3_swap_equal, results_4_swap_equal], group_name="swap_equal")

,Metric,Mean,Std
0,DISTANCE,0.305540,0.130230
1,CONT_DISTANCE,0.401465,0.062184
2,CAT_DISTANCE,0.209615,0.227116
3,SPARSITY,0.493750,0.125018
4,PROCESS_VIOLATION,0.291496,0.237246
5,MARGIN_LOSS,0.587695,0.421999
6,MARGIN_LOSS_NA,0.136504,0.139223
7,MARGIN_LOSS_FLIPPED,0.595833,0.471235
8,MARGIN_LOSS_NA_FLIPPED,0.100000,0.212835
9,FITNESS,1.678480,0.692463


In [33]:
analyze_overall_results([results_1_swap_weighted, results_2_swap_weighted, results_3_swap_weighted, results_4_swap_weighted], group_name="swap_weighted")

,Metric,Mean,Std
0,DISTANCE,0.321787,0.136893
1,CONT_DISTANCE,0.410882,0.066563
2,CAT_DISTANCE,0.232692,0.235962
3,SPARSITY,0.510417,0.128252
4,PROCESS_VIOLATION,0.298393,0.237976
5,MARGIN_LOSS,0.596357,0.434681
6,MARGIN_LOSS_NA,0.131899,0.137440
7,MARGIN_LOSS_FLIPPED,0.608333,0.475242
8,MARGIN_LOSS_NA_FLIPPED,0.091665,0.180806
9,FITNESS,3.516454,2.004098


### --- Cleanup ---

In [34]:
sys.stdout = original_stdout
log_file.close()